In [0]:
from pyspark.sql.functions import to_date, col, split, expr, row_number
from pyspark.sql.window import Window

bronze_schemapath = "subscription_pipeline.bronze"
silver_schemapath = "subscription_pipeline.silver"

In [0]:
# Remove Fivetran ingested columns
fivetran_ingested_column_to_remove = [
    "_file",
    "_fivetran_synced",
    "_modified",
    "_line"
]

def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

def fill_id_column(df, column_name):
    w = Window().orderBy(df.columns[0])
    df = df.withColumn(column_name, row_number().over(w))
    return df

In [0]:
# Read from bronze and remove metadata columns
df = spark.read.table(f"{bronze_schemapath}.product")
df_clean = remove_fivetran_ingested_column(df)

# Add product_id
df_clean = fill_id_column(df_clean, "product_id")

# Convert date columns
df_clean = df_clean.withColumn(
    "created_date",
    to_date(col("created_date"), "dd-MM-yyyy")
)

# Extract product internal ID from name
df_clean = df_clean.withColumn(
    "product_internal_id",
    split(col("product_name"), " ")[2]
)

# Convert is_active to binary
df_clean = df_clean.withColumn(
    "is_active",
    expr("case when is_active = 'Y' then 1 else 0 end")
)

# Convert list_price to double
df_clean = df_clean.withColumn("list_price", col("list_price").cast("double"))

# Remove duplicates
df_clean = df_clean.dropDuplicates()

# Write to silver
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schemapath}.products")

print(f"✓ Products table created: {df_clean.count()} rows")